# Telegram Notifier — Connection Test

## 前置步骤

### 1. 创建 Bot
1. 在 Telegram 搜索 **@BotFather**，发送 `/newbot`
2. 按提示取名，获得 `Bot Token`（格式：`7xxxxxxxxxx:AAxxx...`）

### 2. 获取你的 Chat ID
1. 先给你的 bot 发任意一条消息（不然 `getUpdates` 为空）
2. 运行下方 Cell 2 → 会打印出你的 `chat_id`

### 3. 设置环境变量（推荐，避免明文写在代码里）
```bash
export TELEGRAM_BOT_TOKEN="7xxxxxxxxxx:AAxxx..."
export TELEGRAM_CHAT_ID="123456789"
```
或者直接在 Cell 1 里填入字符串（仅本地测试用）。

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

# ── 填入 Token 和 Chat ID（或留空，从环境变量读取）──
BOT_TOKEN  = os.environ.get("TELEGRAM_BOT_TOKEN",  "")   # 替换为你的 token
CHAT_ID    = os.environ.get("TELEGRAM_CHAT_ID",    "")   # 替换为你的 chat_id

print(f"Token set: {'✓' if BOT_TOKEN else '✗ (missing!)'}")
print(f"Chat ID set: {'✓' if CHAT_ID else '✗ (missing — run Cell 2 first)'}")

In [ ]:
# ── Cell 2: 查询你的 Chat ID ──
# 先给 Bot 发一条任意消息，再运行这个 Cell
import asyncio
from trading.telegram_notifier import TelegramNotifier

assert BOT_TOKEN, "请先填入 BOT_TOKEN"

tmp = TelegramNotifier(token=BOT_TOKEN, chat_id="0")   # chat_id 临时填 0，只查询用
chat_info = await tmp.get_chat_id()
print("Chat info:", chat_info)
print()
print("把上面的 chat_id 填入 Cell 1 的 CHAT_ID 变量，或设置环境变量 TELEGRAM_CHAT_ID")

In [ ]:
# ── Cell 3: 基础连通性测试 ──
from trading.telegram_notifier import TelegramNotifier

assert BOT_TOKEN and CHAT_ID, "请先填入 BOT_TOKEN 和 CHAT_ID"

notifier = TelegramNotifier(token=BOT_TOKEN, chat_id=CHAT_ID)

ok = await notifier.send("✅ <b>连通性测试成功</b>\n来自 <code>mean_reversion_cta_pm</code> Notebook")
print("发送成功：", ok)

In [ ]:
# ── Cell 4: 测试 send_trade（模拟开仓通知）──
ok = await notifier.send_trade(
    action="LONG",
    old_pos=0.0, new_pos=1.0,
    price=152.3400,
    close_pnl_bps=0.0,
    est_fee_bps=2.90,
    est_net_pnl=-2.90,
    cum_pnl_estimated=-2.90,
    symbol="SOL/USDC",
)
print("send_trade:", ok)

In [ ]:
# ── Cell 5: 测试 send_fill（模拟成交回报）──
ok = await notifier.send_fill(
    trade_idx=1,
    fill_status="maker",
    contracts=0.0657,
    avg_fill_price=152.3400,
    actual_fee_bps=2.00,
    est_fee_bps=2.90,
    net_pnl=-2.00,
    cum_pnl_actual=-2.00,
    fill_stats_line="orders=1 maker=1(100%) taker=0 | avg_cost: actual=2.00 est=2.90 slip=-0.90bps",
)
print("send_fill:", ok)

In [ ]:
# ── Cell 6: 测试 send_error（模拟错误通知）──
ok = await notifier.send_error("WebSocket", "Connection reset by peer")
print("send_error:", ok)

In [ ]:
# ── Cell 7: 测试 send_shutdown（模拟关闭汇总）──
ok = await notifier.send_shutdown(
    pnl_estimated=+45.3,
    pnl_actual=+47.1,
    n_trades=5,
    fill_stats_line="orders=5 maker=4(80%) taker=1 | avg_cost: actual=2.20 est=2.90 slip=-0.70bps",
)
print("send_shutdown:", ok)

In [ ]:
# ── Cell 8: from_env() 路径测试（确认环境变量生效）──
# 确保已 export TELEGRAM_BOT_TOKEN 和 TELEGRAM_CHAT_ID

n2 = TelegramNotifier.from_env_optional()
if n2:
    ok = await n2.send("🔧 <b>from_env() 路径测试通过</b>")
    print("from_env notifier send:", ok)
else:
    print("⚠ 环境变量未设置，from_env_optional() 返回 None（符合预期）")